# 3b. Beta Diversity Analysis: MicroOdyssey

## Workflow

1. Import datasets
2. PcoA visualization
3. Statistical testing
4. Longitudinal comparisons
5. (k-mer beta diversity -> do for alpha too!) not sure yet

In [1]:
import os
import pandas as pd
import qiime2 as q2
from skbio import OrdinationResults
from qiime2 import Visualization
from seaborn import scatterplot
from skbio import DistanceMatrix

%matplotlib inline


In [4]:
%cd /home/jovyan/MicroOdyssey/scripts

/home/jovyan/MicroOdyssey/scripts


## 1. Import datasets

In [8]:
data_dir = "../data"
results_dir = "../results/diversity"
core_dir = f"{results_dir}/core-metrics-results"

os.makedirs(results_dir, exist_ok=True)

In [6]:
metadata = pd.read_csv(f"{data_dir}/metadata.csv",sep=";")

metadata.head()

,del,collection_date,country,instrument,sample_id,delivery_mode,host_sex,diet_milk,city,latitude,longitude,urca_code,urca_description,host_id,host_age,seq_depth
0,SRR25052523,19.09.2018,usa,illumina novaseq 6000,1,csection,female,NaN,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),10,2 months,17325587
1,SRR25052512,20.12.2018,usa,illumina novaseq 6000,2,csection,female,mixed,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),10,6 months,12407450
2,SRR25052501,08.07.2019,usa,illumina novaseq 6000,3,csection,female,mixed,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),10,12 months,11964978
3,SRR25052458,30.06.2020,usa,illumina novaseq 6000,4,csection,female,mixed,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),10,24 months,10921526
4,SRR25052720,28.08.2018,usa,illumina novaseq 6000,5,vaginal,male,NaN,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),105,birth,12988703


## 2. PCoA visualization

In [ ]:
! qiime emperor plot \
    --i-pcoa $core_dir/bray_curtis_pcoa_results.qza \
    --m-metadata-file $data_dir/metadata_q2.tsv \
    --o-visualization $core_dir/bray_curtis_emperor.qzv

#-> creates interactive PCoA visualization

In [ ]:
Visualization.load(f"{core_dir}/bray_curtis_emperor.qzv")

In [ ]:
! qiime emperor plot \
    --i-pcoa $core_dir/jaccard_pcoa_results.qza \
    --m-metadata-file $data_dir/metadata_q2.tsv \
    --o-visualization $core_dir/jaccard_emperor.qzv

#-> creates interactive PCoA visualization

In [ ]:
Visualization.load(f"{core_dir}/jaccard_emperor.qzv")

## 3. Statistical testing

### 3.1 PERMANOVA

In [ ]:
#bray-curtis: vaginal vs C-section

! qiime diversity beta-group-significance \
    --i-distance-matrix $core_dir/bray_curtis_distance_matrix.qza \ #input
    --m-metadata-file $data_dir/metadata_q2.tsv \ #metadata
    --m-metadata-column delivery_mode \ #groups
    --p-method permanova \
    --p-permutations 999 \ #randomly reshuffles group labels 999x to test whether observed difference could happen by chance
    --o-visualization $results_dir/bray_delivery_permanova.qzv

In [ ]:
Visualization.load(f"{results_dir}/bray_delivery_permanova.qzv")

In [ ]:
# are result consistent? 
# using jaccard: vaginal vs C-section

! qiime diversity beta-group-significance \
    --i-distance-matrix $core_dir/jaccard_distance_matrix.qza \
    --m-metadata-file $data_dir/metadata_q2.tsv \
    --m-metadata-column delivery_mode \
    --p-method permanova \
    --p-permutations 999 \
    --o-visualization $results_dir/jaccard_delivery_permanova.qzv

Visualization.load(f"{results_dir}/jaccard_delivery_permanova.qzv")

In [ ]:
Visualization.load(f"{results_dir}/jaccard_delivery_permanova.qzv")

### 3.2 PERMDISP

In [ ]:
# bray-curtis: test differences in dispersion
# WHY? PERMANOVA might find a significant difference partly bc one group is much more variable. PERMDISP helps us detect this

! qiime diversity beta-group-significance \
    --i-distance-matrix $core_dir/bray_curtis_distance_matrix.qza \
    --m-metadata-file $data_dir/metadata_q2.tsv \
    --m-metadata-column delivery_mode \
    --p-method permdisp \
    --p-permutations 999 \
    --o-visualization $results_dir/bray_delivery_permdisp.qzv

Visualization.load(f"{results_dir}/bray_delivery_permdisp.qzv")

In [ ]:
Visualization.load(f"{results_dir}/bray_delivery_permdisp.qzv")

In [ ]:
#jaccard: test differences in dispersion

! qiime diversity beta-group-significance \
    --i-distance-matrix $core_dir/jaccard_distance_matrix.qza \
    --m-metadata-file $data_dir/metadata_q2.tsv \
    --m-metadata-column delivery_mode \
    --p-method permdisp \
    --p-permutations 999 \
    --o-visualization $results_dir/jaccard_delivery_permdisp.qzv


In [ ]:
Visualization.load(f"{results_dir}/jaccard_delivery_permdisp.qzv")

### 3.3 Kruskall-Wallis test (alpha diversity)

In [ ]:
! qiime diversity alpha-group-significance \
  --i-alpha-diversity $data_dir/core-metrics-results/shannon_vector.qza \
  --m-metadata-file $data_dir/metadata_q2.tsv \
  --o-visualization $data_dir/core-metrics-results/shannon-group-significance.qzv

In [ ]:
Visualization.load(f"{data_dir}/core-metrics-results/shannon-group-significance.qzv")

In [ ]:
! qiime diversity alpha-correlation \
  --i-alpha-diversity $data_dir/core-metrics-results/shannon_vector.qza \
  --m-metadata-file $data_dir/metadata_q2.tsv \
  --o-visualization $data_dir/core-metrics-results/shannon-group-significance-numeric.qzv

In [ ]:
Visualization.load(f"{data_dir}/core-metrics-results/shannon-group-significance-numeric.qzv")